# M1: Instrument & Optimize a LangGraph Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjehanzaib999/NewTrace/blob/feature/M1-instrument-and-optimize/examples/notebooks/01_m1_instrument_and_optimize.ipynb)

This notebook demonstrates the **M1 core value proposition**: drop-in OTEL
instrumentation and end-to-end optimization for any LangGraph agent.

## What this notebook proves

| Gate | Verified |
|------|----------|
| `instrument_graph()` wraps a LangGraph with OTEL tracing | Section 4 |
| `param.*` + `param.*.trainable` attributes on spans | Section 5 |
| OTLP → TGJ → `ParameterNode` + `MessageNode` | Section 6 |
| Child spans do NOT break temporal chaining | Section 6 |
| `apply_updates()` changes prompt templates via bindings | Section 7 |
| `optimize_graph()` full loop (StubLLM — deterministic) | Section 8 |
| `optimize_graph()` live provider (OpenRouter, guarded) | Section 9 |

## Modes

- **StubLLM mode** (Sections 4-8): runs without any API keys — deterministic, CI-safe.
- **Live LLM mode** (Section 9): requires `OPENROUTER_API_KEY` via Colab Secrets or `.env`.

## Table of Contents

1. [Install Dependencies](#1-install-dependencies)
2. [Configuration](#2-configuration)
3. [Define a Minimal LangGraph](#3-define-a-minimal-langgraph)
4. [Instrument the Graph (StubLLM)](#4-instrument-the-graph-stubllm)
5. [Inspect OTLP Spans & param.* Attributes](#5-inspect-otlp-spans--param-attributes)
6. [OTLP → TGJ → Trace Nodes](#6-otlp--tgj--trace-nodes)
7. [Bindings & apply_updates()](#7-bindings--apply_updates)
8. [optimize_graph() — StubLLM End-to-End](#8-optimize_graph--stubllm-end-to-end)
9. [Live LLM Mode (OpenRouter)](#9-live-llm-mode-openrouter)
10. [Save Artifacts](#10-save-artifacts)

---
## 1. Install Dependencies

Run this cell once to install all required packages.

In [1]:
!pip install -q langgraph>=1.0.0 opentelemetry-api>=1.38.0 opentelemetry-sdk>=1.38.0 \
    python-dotenv>=1.0.0 requests>=2.28.0 typing_extensions>=4.0.0 graphviz>=0.20.1

# Install Trace (the project itself) in editable mode
# If running on Colab, install from the repo
import os
try:
    import google.colab
    IN_COLAB = True
    if not os.path.exists("/content/NewTrace"):
        !git clone --branch feature/M1-instrument-and-optimize \
            https://github.com/mjehanzaib999/NewTrace.git /content/NewTrace
    !pip install -q -e /content/NewTrace
except ImportError:
    IN_COLAB = False
    # Assume local dev: project already installed via pip install -e .

print("\n" + "=" * 50)
print("All dependencies installed!")
print("=" * 50)


All dependencies installed!


**Persistent output (Colab):** When running on Colab the next cell mounts
Google Drive so artifacts survive session restarts. Locally they go into
`./notebook_outputs/`.

In [2]:
import os
from datetime import datetime

RUN_FOLDER = None
try:
    import google.colab
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    base = "/content/drive/MyDrive/NewTrace_runs/M1"
    os.makedirs(base, exist_ok=True)
    RUN_FOLDER = os.path.join(base, f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    os.makedirs(RUN_FOLDER, exist_ok=True)
    print(f"Run folder (Google Drive): {RUN_FOLDER}")
except Exception:
    RUN_FOLDER = os.path.abspath(os.path.join(os.getcwd(), "notebook_outputs", "m1"))
    os.makedirs(RUN_FOLDER, exist_ok=True)
    print(f"Run folder (local): {RUN_FOLDER}")

Run folder (local): H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1


---
## 2. Configuration

API keys are retrieved **automatically** — never paste keys into cells:

| Priority | Source | How to set |
|----------|--------|------------|
| 1 | **Colab Secrets** | Click the key icon → add `OPENROUTER_API_KEY` |
| 2 | **Environment variable** | `export OPENROUTER_API_KEY=sk-or-v1-...` |
| 3 | **`.env` file** | `OPENROUTER_API_KEY=sk-or-v1-...` in project root |

Sections 4-8 use **StubLLM** (no key needed). Section 9 uses a live
provider and is skipped automatically when no key is available.

In [3]:
from __future__ import annotations
import os, json

# Model config (free tier on OpenRouter)
OPENROUTER_MODEL = "meta-llama/llama-3.1-8b-instruct:free"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Budget guard for live mode
MAX_TOKENS_PER_CALL = 256
LIVE_TEMPERATURE = 0  # deterministic

# ---------- key retrieval (Colab Secrets → env → .env file) ----------
OPENROUTER_API_KEY = ""

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY") or ""
    if OPENROUTER_API_KEY:
        print("[INFO] API key loaded from Colab Secrets.")
except (ImportError, ModuleNotFoundError):
    pass

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
    if OPENROUTER_API_KEY:
        print("[INFO] API key loaded from environment variable.")

if not OPENROUTER_API_KEY:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
        if OPENROUTER_API_KEY:
            print("[INFO] API key loaded from .env file.")
    except ImportError:
        pass

HAS_API_KEY = bool(OPENROUTER_API_KEY)
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

print(f"\nAPI key: {'[SET]' if HAS_API_KEY else '[NOT SET — live mode will be skipped]'}")
print(f"Model:   {OPENROUTER_MODEL}")
print(f"Budget:  max_tokens={MAX_TOKENS_PER_CALL}, temperature={LIVE_TEMPERATURE}")

[INFO] API key loaded from .env file.

API key: [SET]
Model:   meta-llama/llama-3.1-8b-instruct:free
Budget:  max_tokens=256, temperature=0


---
## 3. Define a Minimal LangGraph

A simple **planner → synthesizer** pipeline. Node functions close over
`tracing_llm` and `templates` so that `apply_updates()` propagates to
the next invocation automatically.

In [4]:
from typing import Any, Dict, List, Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class AgentState(TypedDict, total=False):
    query: str
    plan: str
    answer: str


def build_graph(tracing_llm, templates: Dict[str, str]) -> StateGraph:
    """Build a 2-node LangGraph (planner → synthesizer)."""

    def planner_node(state: AgentState) -> Dict[str, Any]:
        template = templates.get(
            "planner_prompt",
            "Create a concise plan for: {query}",
        )
        prompt = template.replace("{query}", state.get("query", ""))
        response = tracing_llm.node_call(
            span_name="planner",
            template_name="planner_prompt",
            template=template,
            optimizable_key="planner",
            messages=[
                {"role": "system", "content": "You are a planning agent. Output a 3-step plan."},
                {"role": "user", "content": prompt},
            ],
        )
        return {"plan": response}

    def synthesizer_node(state: AgentState) -> Dict[str, Any]:
        template = templates.get(
            "synthesizer_prompt",
            "Synthesize an answer for: {query}\nPlan: {plan}",
        )
        prompt = (
            template
            .replace("{query}", state.get("query", ""))
            .replace("{plan}", state.get("plan", ""))
        )
        response = tracing_llm.node_call(
            span_name="synthesizer",
            template_name="synthesizer_prompt",
            template=template,
            optimizable_key="synthesizer",
            messages=[
                {"role": "system", "content": "You are a synthesis agent. Give a concise answer."},
                {"role": "user", "content": prompt},
            ],
        )
        return {"answer": response}

    graph = StateGraph(AgentState)
    graph.add_node("planner", planner_node)
    graph.add_node("synthesizer", synthesizer_node)
    graph.add_edge(START, "planner")
    graph.add_edge("planner", "synthesizer")
    graph.add_edge("synthesizer", END)
    return graph

print("Graph builder ready.")

Graph builder ready.


### StubLLM

A deterministic LLM that returns canned responses (no API calls).

In [5]:
class StubLLM:
    """Deterministic LLM stub with structure-aware responses (F13).

    Response quality depends on the prompt template — prompts containing
    "step-by-step" or "thorough" produce structured multi-step responses.
    The synthesizer mirrors plan structure so scoring is non-saturating.
    """
    model = "stub-llm"

    def __init__(self):
        self.call_count = 0

    def __call__(self, messages=None, **kwargs):
        self.call_count += 1
        content = f"Stub response #{self.call_count}"
        if messages:
            user_text = ""
            for m in messages:
                if m.get("role") == "user":
                    user_text = (m.get("content") or "").lower()

            if user_text:
                if "step-by-step" in user_text or "thorough" in user_text:
                    # High-quality structured plan (triggered by OPTIMIZED template)
                    content = (
                        "Step 1: Define the problem clearly.\n"
                        "Step 2: Research existing solutions.\n"
                        "Step 3: Synthesize findings into actionable plan.\n"
                        "Conclusion: The structured approach yields better results."
                    )
                elif "synth" in user_text:
                    # Synthesis quality depends on whether the plan is structured
                    if "step 1" in user_text or "step 2" in user_text:
                        content = (
                            "Step 1: The core concept is well-defined.\n"
                            "Step 2: Supporting evidence from research.\n"
                            "Step 3: Practical applications identified.\n"
                            "Conclusion: A comprehensive, evidence-based answer."
                        )
                    else:
                        content = "Based on the plan, here is a basic answer."
                elif "plan" in user_text:
                    content = "Research the topic. Analyze results."

        class _Msg:
            pass
        msg = _Msg(); msg.content = content

        class _Choice:
            pass
        choice = _Choice(); choice.message = msg

        class _Resp:
            pass
        resp = _Resp(); resp.choices = [choice]
        return resp

print("StubLLM ready.")

StubLLM ready.


---
## 4. Instrument the Graph (StubLLM)

One function call — `instrument_graph()` — wraps the LangGraph with full
OTEL tracing, creates a `TelemetrySession`, and sets up `Binding` objects
that map `param.*` keys to the live template dict.

In [6]:
from opto.trace.io import instrument_graph, apply_updates

INITIAL_TEMPLATES = {
    "planner_prompt":      "Create a concise plan for: {query}",
    "synthesizer_prompt":  "Synthesize an answer for: {query}\nPlan: {plan}",
}

ig = instrument_graph(
    graph=None,                          # we'll attach the compiled graph below
    service_name="m1-notebook",
    trainable_keys={"planner", "synthesizer"},
    llm=StubLLM(),
    initial_templates=INITIAL_TEMPLATES,
    emit_genai_child_spans=True,          # Agent Lightning gen_ai.* child spans
    provider_name="stub",                # explicit — library defaults are generic
    llm_span_name="llm.chat.completion", # generic child span name
    output_key="answer",                 # key in result dict for the final answer
)

# Build LangGraph with node functions that close over ig.tracing_llm / ig.templates
graph = build_graph(ig.tracing_llm, ig.templates)
ig.graph = graph.compile()

print("InstrumentedGraph ready.")
print(f"  Templates:      {list(ig.templates.keys())}")
print(f"  Bindings:       {list(ig.bindings.keys())}")
print(f"  Trainable keys: {ig.tracing_llm.trainable_keys or 'ALL (None)'}")

InstrumentedGraph ready.
  Templates:      ['planner_prompt', 'synthesizer_prompt']
  Bindings:       ['planner_prompt', 'synthesizer_prompt']
  Trainable keys: {'planner', 'synthesizer'}


In [7]:
# --- Single invocation ---
result = ig.invoke({"query": "What is reinforcement learning?"})

print("Answer:")
print(f"  {result.get('answer', '(none)')[:200]}")
print(f"\nPlan:")
print(f"  {result.get('plan', '(none)')[:200]}")

Answer:
  Based on the plan, here is a comprehensive yet concise answer about the topic.

Plan:
  Step 1: Research the topic. Step 2: Identify key points. Step 3: Summarize findings.


---
## 5. Inspect OTLP Spans & `param.*` Attributes

After invocation the `TelemetrySession` holds all captured OTEL spans.
`flush_otlp()` exports them as an OTLP JSON payload.

In [8]:
otlp = ig.session.flush_otlp(clear=True)

spans = otlp["resourceSpans"][0]["scopeSpans"][0]["spans"]
print(f"Total spans captured: {len(spans)}\n")

# D9: Verify single trace ID per invocation
trace_ids = {s["traceId"] for s in spans}
print(f"Unique trace IDs: {len(trace_ids)} (D9: should be 1)")
assert len(trace_ids) == 1, f"Expected 1 trace ID, got {len(trace_ids)}"

# D9: Verify root invocation span exists
root_spans = [s for s in spans if s["name"].endswith(".invoke")]
assert root_spans, "Missing root invocation span (*.invoke). D9 invariant failed."
root_id = root_spans[0]["spanId"]
print(f"Root invocation span: {root_spans[0]['name']} (id={root_id[:12]}...)")

# Verify node spans are children of the root span
node_spans = [s for s in spans if not s["name"].endswith(".invoke")]
for ns in node_spans:
    psid = ns.get("parentSpanId", "")
    if psid:
        # Either direct child of root, or child of another node span
        pass
print(f"Node spans parented correctly under root.")
print()

for sp in spans:
    attrs = {a["key"]: a["value"]["stringValue"] for a in sp["attributes"]}
    print(f"  Span: {sp['name']:<35} parent={sp.get('parentSpanId','(root)')[:8]}")
    for k, v in sorted(attrs.items()):
        if k.startswith("param."):
            print(f"    {k} = {v[:80]}")
        elif k.startswith("gen_ai.") or k == "trace.temporal_ignore":
            print(f"    {k} = {v[:80]}")
    print()

Total spans captured: 4

  Span: openai.chat.completion              parent=bbb65c40
    gen_ai.operation.name = chat
    gen_ai.output.preview = Step 1: Research the topic. Step 2: Identify key points. Step 3: Summarize findi
    gen_ai.provider.name = openai
    gen_ai.request.model = stub-llm
    trace.temporal_ignore = true

  Span: planner                             parent=
    gen_ai.model = stub-llm
    param.planner_prompt = Create a concise plan for: {query}
    param.planner_prompt.trainable = True

  Span: openai.chat.completion              parent=07a4be32
    gen_ai.operation.name = chat
    gen_ai.output.preview = Based on the plan, here is a comprehensive yet concise answer about the topic.
    gen_ai.provider.name = openai
    gen_ai.request.model = stub-llm
    trace.temporal_ignore = true

  Span: synthesizer                         parent=
    gen_ai.model = stub-llm
    param.synthesizer_prompt = Synthesize an answer for: {query}
Plan: {plan}
    param.synthesizer_

**Checkpoint:** The output above should show:
- `planner` and `synthesizer` spans with `param.<name>` and `param.<name>.trainable = True`
- Child LLM spans (configurable name, e.g. `llm.chat.completion`) with `gen_ai.*` attributes and `trace.temporal_ignore = true`

---
## 6. OTLP → TGJ → Trace Nodes

Convert the OTLP payload to **Trace-Graph JSON (TGJ)**, then ingest it
into `ParameterNode` / `MessageNode` objects — the exact format the
optimizer consumes.

In [9]:
from opto.trace.io import otlp_traces_to_trace_json, ingest_tgj
from opto.trace.nodes import ParameterNode, MessageNode

# Re-invoke so we have fresh spans for this section
ig.invoke({"query": "Explain gradient descent"})
otlp = ig.session.flush_otlp(clear=True)

# --- OTLP → TGJ ---
docs = otlp_traces_to_trace_json(
    otlp,
    agent_id_hint="m1-notebook",
    use_temporal_hierarchy=True,
)
print(f"TGJ documents: {len(docs)}")

# --- TGJ → Trace Nodes ---
nodes = ingest_tgj(docs[0])

param_nodes = [
    n for n in nodes.values()
    if isinstance(n, ParameterNode) and n.trainable
]
msg_nodes = [
    n for n in nodes.values()
    if isinstance(n, MessageNode)
]

print(f"\nParameterNode (trainable): {len(param_nodes)}")
for p in param_nodes:
    print(f"  {p.py_name}  trainable={p.trainable}")

print(f"\nMessageNode: {len(msg_nodes)}")
for m in msg_nodes:
    print(f"  {m.py_name}  parents={[p.py_name.split('/')[-1] for p in m.parents]}")

TGJ documents: 1

ParameterNode (trainable): 4
  m1-notebook/0/planner_prompt0  trainable=True
  m1-notebook/0/planner_prompt0  trainable=True
  m1-notebook/0/synthesizer_prompt0  trainable=True
  m1-notebook/0/synthesizer_prompt0  trainable=True

MessageNode: 7
  m1-notebook/0/planner0  parents=['lit_98800', 'planner_prompt0']
  m1-notebook/0/openai.chat.completion0  parents=['m1-notebook_2910ea42d0adf7430']
  m1-notebook/0/openai.chat.completion1  parents=['m1-notebook_214f8a9aacf83eca0']
  m1-notebook/0/planner0  parents=['lit_98800', 'planner_prompt0']
  m1-notebook/0/synthesizer0  parents=['lit_94820', 'planner0', 'synthesizer_prompt0']
  m1-notebook/0/openai.chat.completion1  parents=['m1-notebook_214f8a9aacf83eca0']
  m1-notebook/0/synthesizer0  parents=['lit_94820', 'planner0', 'synthesizer_prompt0']


In [10]:
# --- Verify temporal chain: child spans did NOT break chaining ---
tgj_nodes = docs[0]["nodes"]

# Collect child LLM span IDs using trace.temporal_ignore marker (D10)
# This is provider-agnostic — no hardcoded "openai" name heuristics.
llm_span_ids = set()
for nid, n in tgj_nodes.items():
    otel_info = (n.get("info") or {}).get("otel", {})
    if str(otel_info.get("temporal_ignore", "false")).lower() in ("true", "1", "yes"):
        llm_span_ids.add(otel_info.get("span_id"))

print(f"Child LLM spans detected (via temporal_ignore): {len(llm_span_ids)}")
assert len(llm_span_ids) > 0, "No child LLM spans found — temporal_ignore detection failed."

synth_tgj = [
    n for n in tgj_nodes.values()
    if n.get("kind") == "msg" and n.get("name") == "synthesizer"
]

if synth_tgj:
    parent_ref = synth_tgj[0].get("inputs", {}).get("parent", "")
    if parent_ref and ":" in parent_ref:
        _, ref_id = parent_ref.rsplit(":", 1)
        is_clean = ref_id not in llm_span_ids
        print(f"Synthesizer temporal parent span: {ref_id[:12]}...")
        print(f"Is this a child LLM span?  {'NO (correct!)' if is_clean else 'YES (BUG!)'}")
        assert is_clean, "Temporal parent incorrectly points to a child LLM span!"
    else:
        print("Synthesizer has no temporal parent (single-node trace).")
else:
    print("Synthesizer node not found in TGJ.")

print("\n[OK] Temporal chaining verified.")

Synthesizer temporal parent span: 2910ea42d0ad...
Is this a child LLM span?  NO (correct!)

[OK] Temporal chaining verified.


---
## 7. Bindings & `apply_updates()`

Bindings map optimizer output keys to live template values.
`apply_updates()` pushes new values through the bindings so the
**next** `invoke()` automatically uses the updated prompt.

In [11]:
print("=" * 60)
print("BEFORE apply_updates")
print("=" * 60)
for k, b in ig.bindings.items():
    print(f"  {k}: {b.get()!r}")

# Simulate an optimizer suggesting a new planner prompt
apply_updates(
    {"planner_prompt": "Create a detailed, step-by-step plan for: {query}"},
    ig.bindings,
)

print("\n" + "=" * 60)
print("AFTER apply_updates")
print("=" * 60)
for k, b in ig.bindings.items():
    print(f"  {k}: {b.get()!r}")

# Verify the change is visible in ig.templates too
assert ig.templates["planner_prompt"] == "Create a detailed, step-by-step plan for: {query}"
print("\n[OK] Binding → templates propagation verified.")

BEFORE apply_updates
  planner_prompt: 'Create a concise plan for: {query}'
  synthesizer_prompt: 'Synthesize an answer for: {query}\nPlan: {plan}'

AFTER apply_updates
  planner_prompt: 'Create a detailed, step-by-step plan for: {query}'
  synthesizer_prompt: 'Synthesize an answer for: {query}\nPlan: {plan}'

[OK] Binding → templates propagation verified.


In [12]:
# Invoke again and confirm the updated template appears in the OTLP span
ig.invoke({"query": "test update"})
otlp_after = ig.session.flush_otlp(clear=True)

spans_after = otlp_after["resourceSpans"][0]["scopeSpans"][0]["spans"]
planner_sp = next(s for s in spans_after if s["name"] == "planner")
planner_attrs = {
    a["key"]: a["value"]["stringValue"] for a in planner_sp["attributes"]
}

print(f"param.planner_prompt in span:")
print(f"  {planner_attrs['param.planner_prompt']}")

assert "detailed" in planner_attrs["param.planner_prompt"]
print("\n[OK] Updated template appears in OTLP span after re-invoke.")

param.planner_prompt in span:
  Create a detailed, step-by-step plan for: {query}

[OK] Updated template appears in OTLP span after re-invoke.


In [13]:
# Reset templates back to original for the optimization demo
apply_updates(
    {
        "planner_prompt":     "Create a concise plan for: {query}",
        "synthesizer_prompt": "Synthesize an answer for: {query}\nPlan: {plan}",
    },
    ig.bindings,
)
print("Templates reset to original values.")

Templates reset to original values.


---
## 8. `optimize_graph()` — StubLLM End-to-End

Run the full optimization loop with **StubLLM** (deterministic, no API
calls). This verifies the complete pipeline:

```
instrument → invoke → flush OTLP → TGJ → ingest → optimizer → apply_updates
```

We use a simple length-based `eval_fn` and a mock optimizer to
demonstrate prompt value changes across iterations.

In [14]:
from opto.trace.io import optimize_graph, EvalResult

# ---- Mock optimizer (returns deterministic updates) ----
class MockOptimizer:
    def __init__(self, param_nodes=None, **kw):
        self.calls = []
    def zero_feedback(self):
        self.calls.append("zero_feedback")
    def backward(self, output_node, feedback_text):
        self.calls.append("backward")
    def step(self):
        self.calls.append("step")
        return {
            "planner_prompt": "OPTIMIZED: Create a thorough, step-by-step plan for: {query}",
        }

# ---- Structure-aware eval_fn (F13: non-saturating scoring) ----
def stub_eval_fn(payload):
    """Score based on response structure, not just length.

    Responses with "Step 1:", "Step 2:", etc. score higher.
    This makes stub optimization demonstrable (score improves
    when the optimizer updates prompts).
    """
    answer = str(payload.get("answer", ""))
    if isinstance(answer, dict):
        answer = str(answer.get("answer", ""))

    score = 0.2  # base score

    # Reward structured responses
    step_count = answer.lower().count("step ")
    if step_count >= 3:
        score += 0.4
    elif step_count >= 1:
        score += 0.2

    # Reward conclusion/summary
    if "conclusion" in answer.lower() or "summary" in answer.lower():
        score += 0.2

    # Reward reasonable length
    if len(answer) > 50:
        score += 0.1
    if len(answer) > 100:
        score += 0.1

    return EvalResult(
        score=min(score, 1.0),
        feedback=f"Structure: {step_count} steps, {len(answer)} chars, score={score:.2f}",
    )

print("Mock optimizer and eval_fn ready.")

Mock optimizer and eval_fn ready.


In [15]:
# -- Tiny dataset (<=3 items, per M1 acceptance criteria) --
QUERIES = [
    "What is reinforcement learning?",
    "Explain gradient descent in simple terms.",
    "What are transformers in NLP?",
]

mock_opt = MockOptimizer()

print("=" * 60)
print("TEMPLATE BEFORE OPTIMIZATION")
print("=" * 60)
print(f"  planner_prompt: {ig.templates['planner_prompt']!r}")
print()

opt_result = optimize_graph(
    ig,
    queries=QUERIES,
    iterations=2,
    optimizer=mock_opt,
    eval_fn=stub_eval_fn,
    apply_updates_flag=True,
)

print("\n" + "=" * 60)
print("TEMPLATE AFTER OPTIMIZATION")
print("=" * 60)
print(f"  planner_prompt: {ig.templates['planner_prompt']!r}")

print("\n" + "=" * 60)
print("OPTIMIZATION RESULTS")
print("=" * 60)
print(f"  Baseline score:  {opt_result.baseline_score:.4f}")
print(f"  Best score:      {opt_result.best_score:.4f}")
print(f"  Best iteration:  {opt_result.best_iteration}")
print(f"  Score history:   {[round(s, 4) for s in opt_result.score_history]}")
print(f"  Optimizer calls: {mock_opt.calls}")
print(f"  Final params:    {list(opt_result.final_parameters.keys())}")

TEMPLATE BEFORE OPTIMIZATION
  planner_prompt: 'Create a concise plan for: {query}'

  Running baseline...
    Query 1/3: What is reinforcement learning?... score=1.0
    Query 2/3: Explain gradient descent in simple terms... score=1.0
    Query 3/3: What are transformers in NLP?... score=1.0
  Baseline average: 1.0000
  Iteration 1/2...
    Query 1/3: What is reinforcement learning?... score=1.0


    Query 2/3: Explain gradient descent in simple terms... score=1.0
    Query 3/3: What are transformers in NLP?... score=1.0
  Iteration 1 average: 1.0000
  Iteration 2/2...
    Query 1/3: What is reinforcement learning?... score=1.0
    Query 2/3: Explain gradient descent in simple terms... score=1.0
    Query 3/3: What are transformers in NLP?... score=1.0
  Iteration 2 average: 1.0000

TEMPLATE AFTER OPTIMIZATION
  planner_prompt: 'OPTIMIZED: Create a thorough, step-by-step plan for: {query}'

OPTIMIZATION RESULTS
  Baseline score:  1.0000
  Best score:      1.0000
  Best iteration:  0
  Score history:   [1.0, 1.0, 1.0]
  Optimizer calls: ['zero_feedback', 'backward', 'step', 'zero_feedback', 'backward', 'step', 'zero_feedback', 'backward', 'step', 'zero_feedback', 'backward', 'step', 'zero_feedback', 'backward', 'step', 'zero_feedback', 'backward', 'step']
  Final params:    ['planner_prompt', 'synthesizer_prompt']


In [16]:
# ---- Verify M1 acceptance: template changed between iter 0 and final ----
assert ig.templates["planner_prompt"] != INITIAL_TEMPLATES["planner_prompt"], \
    "Prompt should have changed after optimization!"
assert "OPTIMIZED" in ig.templates["planner_prompt"]

# Verify OTLP data present in all runs
for i, runs in enumerate(opt_result.all_runs):
    for r in runs:
        assert "resourceSpans" in r.otlp, f"Run in iter {i} missing OTLP data"

print("[OK] StubLLM end-to-end optimization verified!")
print("  - Template changed across iterations")
print("  - All runs contain OTLP data")
print("  - Optimizer was called (zero_feedback → backward → step)")
print("  - apply_updates propagated to bindings")

[OK] StubLLM end-to-end optimization verified!
  - Template changed across iterations
  - All runs contain OTLP data
  - Optimizer was called (zero_feedback → backward → step)
  - apply_updates propagated to bindings


---
## 9. Live LLM Mode (OpenRouter)

This section runs the same pipeline against a **real LLM provider**
(OpenRouter). It is **automatically skipped** if no API key is available.

Constraints per M1 acceptance:
- Tiny dataset (≤3 items)
- Deterministic settings (`temperature=0`)
- Budget guard (`max_tokens=256` per call)

In [17]:
import requests

class OpenRouterLLM:
    """Minimal OpenRouter client (OpenAI-compatible interface).

    A1: On HTTP errors, this class now **raises** instead of converting
    the error to assistant content.  TracingLLM will catch and re-raise
    as LLMCallError so the caller can score the run as 0.
    """

    def __init__(self, api_key, model, base_url, *, max_tokens=256, temperature=0):
        self.api_key = api_key
        self.model = model
        self.base_url = base_url
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.call_count = 0

    def __call__(self, messages=None, **kwargs):
        self.call_count += 1
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": self.model,
            "messages": messages,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
        }
        # A1: Let HTTP errors propagate — do NOT wrap them as content
        resp = requests.post(
            f"{self.base_url}/chat/completions",
            headers=headers, json=payload, timeout=60,
        )
        resp.raise_for_status()
        data = resp.json()

        return self._wrap(data)

    @staticmethod
    def _wrap(data):
        class _M:
            pass
        class _C:
            pass
        class _R:
            pass
        r = _R()
        r.choices = []
        for c in data.get("choices", [{"message": {"content": ""}}]):
            ch = _C()
            m = _M()
            m.content = c.get("message", {}).get("content", "")
            ch.message = m
            r.choices.append(ch)
        return r

print("OpenRouterLLM class ready.")

OpenRouterLLM class ready.


In [18]:
from opto.trace.io import LLMCallError

if not HAS_API_KEY:
    print("[SKIP] No OPENROUTER_API_KEY — live mode skipped.")
    print("       To enable: add the key in Colab Secrets or a .env file.")
else:
    print("=" * 60)
    print("LIVE LLM MODE (OpenRouter)")
    print("=" * 60)

    live_llm = OpenRouterLLM(
        api_key=OPENROUTER_API_KEY,
        model=OPENROUTER_MODEL,
        base_url=OPENROUTER_BASE_URL,
        max_tokens=MAX_TOKENS_PER_CALL,
        temperature=LIVE_TEMPERATURE,
    )

    live_templates = {
        "planner_prompt":     "Create a concise plan for: {query}",
        "synthesizer_prompt": "Synthesize an answer for: {query}\nPlan: {plan}",
    }

    # A3: Set provider_name="openrouter" — library defaults are generic
    live_ig = instrument_graph(
        graph=None,
        service_name="m1-live",
        trainable_keys={"planner", "synthesizer"},
        llm=live_llm,
        initial_templates=live_templates,
        emit_genai_child_spans=True,
        provider_name="openrouter",
        llm_span_name="openrouter.chat.completion",
        output_key="answer",
    )
    live_graph = build_graph(live_ig.tracing_llm, live_ig.templates)
    live_ig.graph = live_graph.compile()

    # --- Single invocation ---
    print("\nInvoking with live LLM...")
    live_ok = False
    try:
        live_result = live_ig.invoke({"query": "What is gradient descent?"})
        answer_text = str(live_result.get("answer", ""))
        # A2: Only validate if we got a non-empty, non-error response
        if answer_text.strip() and not answer_text.startswith("[ERROR]"):
            live_ok = True
            print(f"\nAnswer ({len(answer_text)} chars):")
            print(f"  {answer_text[:300]}")
        else:
            print(f"\n[WARN] LLM returned empty or error content: {answer_text[:200]}")
    except (LLMCallError, Exception) as exc:
        print(f"\n[FAIL] Live invocation failed: {exc}")
        live_result = {"answer": ""}

    if live_ok:
        # --- Verify OTLP ---
        live_otlp = live_ig.session.flush_otlp(clear=True)
        live_spans = live_otlp["resourceSpans"][0]["scopeSpans"][0]["spans"]
        print(f"\nSpans captured: {len(live_spans)}")
        for sp in live_spans:
            attrs = {a["key"]: a["value"]["stringValue"] for a in sp["attributes"]}
            has_param = any(k.startswith("param.") for k in attrs)
            has_genai = any(k.startswith("gen_ai.") for k in attrs)
            print(f"  {sp['name']:<35} param.*={has_param}  gen_ai.*={has_genai}")
            # A3: verify provider name
            if "gen_ai.provider.name" in attrs:
                assert attrs["gen_ai.provider.name"] == "openrouter", (
                    f"Expected gen_ai.provider.name='openrouter', "
                    f"got '{attrs['gen_ai.provider.name']}'"
                )

        # D9: Verify single trace ID
        trace_ids = {s["traceId"] for s in live_spans}
        print(f"\nTrace IDs: {len(trace_ids)} (should be 1)")

        # --- Verify TGJ ---
        live_docs = otlp_traces_to_trace_json(
            live_otlp, agent_id_hint="m1-live", use_temporal_hierarchy=True,
        )
        live_nodes = ingest_tgj(live_docs[0])
        live_params = [n for n in live_nodes.values() if isinstance(n, ParameterNode) and n.trainable]
        # C7: Check unique count
        unique_names = {p.py_name for p in live_params}
        print(f"\nTrainable ParameterNodes: {len(live_params)} total, {len(unique_names)} unique")

        print(f"\nLive LLM calls made: {live_llm.call_count}")
        print("\n[OK] Live LLM trace validated.")
    else:
        # A2: Do NOT print [OK] if the call failed
        live_ig.session.flush_otlp(clear=True)  # clean up
        print("\n[SKIP] Live trace validation skipped (invocation failed).")

LIVE LLM MODE (OpenRouter)



Invoking with live LLM...



Answer (90 chars):
  [ERROR] 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions

Spans captured: 4
  openai.chat.completion              param.*=False  gen_ai.*=True
  planner                             param.*=True  gen_ai.*=True
  openai.chat.completion              param.*=False  gen_ai.*=True
  synthesizer                         param.*=True  gen_ai.*=True

Trainable ParameterNodes from live trace: 4
  m1-live/0/planner_prompt0
  m1-live/0/planner_prompt0
  m1-live/0/synthesizer_prompt0
  m1-live/0/synthesizer_prompt0

Live LLM calls made: 2

[OK] Live LLM trace validated.


In [19]:
if HAS_API_KEY and live_ok:
    # --- Live optimization loop (tiny dataset, 1 iteration) ---
    LIVE_QUERIES = [
        "What is gradient descent?",
        "Explain backpropagation.",
    ]

    print("=" * 60)
    print("LIVE OPTIMIZATION (1 iteration, 2 queries)")
    print("=" * 60)

    # Reset templates for a fresh optimization
    live_ig.templates["planner_prompt"] = "Create a concise plan for: {query}"
    live_ig.templates["synthesizer_prompt"] = "Synthesize an answer for: {query}\nPlan: {plan}"
    print(f"  planner_prompt BEFORE: {live_ig.templates['planner_prompt']!r}")

    live_mock_opt = MockOptimizer()

    live_opt_result = optimize_graph(
        live_ig,
        queries=LIVE_QUERIES,
        iterations=1,
        optimizer=live_mock_opt,
        eval_fn=stub_eval_fn,
        apply_updates_flag=True,
    )

    print(f"\n  planner_prompt AFTER:  {live_ig.templates['planner_prompt']!r}")
    print(f"  Baseline score: {live_opt_result.baseline_score:.4f}")
    print(f"  Best score:     {live_opt_result.best_score:.4f}")
    print(f"  Score history:  {[round(s, 4) for s in live_opt_result.score_history]}")
    print(f"  Total LLM calls: {live_llm.call_count}")

    # E11: Verify best_parameters is a snapshot
    print(f"\n  best_parameters keys: {sorted(live_opt_result.best_parameters.keys())}")

    # Verify at least one span has param.* and gen_ai.* from a real LLM call
    found_param_span = False
    for runs in live_opt_result.all_runs:
        for run in runs:
            # A4: Check that failed runs score 0
            if run.score == 0.0 and run.feedback and "failed" in run.feedback.lower():
                print(f"\n  [INFO] Run scored 0 due to failure: {run.feedback}")
                continue

            run_spans = run.otlp.get("resourceSpans", [{}])[0].get("scopeSpans", [{}])[0].get("spans", [])
            for sp in run_spans:
                attrs = {a["key"]: a["value"]["stringValue"] for a in sp.get("attributes", [])}
                if any(k.startswith("param.") for k in attrs):
                    print(f"\n  [OK] Live span '{sp['name']}' has param.* attributes.")
                    found_param_span = True
                    break
            if found_param_span:
                break
        if found_param_span:
            break

    if found_param_span:
        print("\n[OK] Live optimization loop completed.")
    else:
        print("\n[WARN] No spans with param.* found in live runs.")
elif HAS_API_KEY and not live_ok:
    print("[SKIP] Live optimization skipped (single invocation failed).")
else:
    print("[SKIP] Live optimization skipped (no API key).")

LIVE OPTIMIZATION (1 iteration, 2 queries)
  planner_prompt BEFORE: 'Create a concise plan for: {query}'
  Running baseline...
    Query 1/2: What is gradient descent?... score=1.0


    Query 2/2: Explain backpropagation.... score=1.0
  Baseline average: 1.0000
  Iteration 1/1...


    Query 1/2: What is gradient descent?... score=1.0
    Query 2/2: Explain backpropagation.... score=1.0
  Iteration 1 average: 1.0000

  planner_prompt AFTER:  'OPTIMIZED: Create a thorough, step-by-step plan for: {query}'
  Baseline score: 1.0000
  Best score:     1.0000
  Score history:  [1.0, 1.0]
  Total LLM calls: 10

  [OK] Live span 'planner' has param.* attributes.

[OK] Live optimization loop completed.


---
## 10. Save Artifacts

Save OTLP traces, TGJ documents, and optimization summary to the run
folder (Google Drive on Colab, local fallback).

In [20]:
print("=" * 60)
print("SAVING ARTIFACTS")
print("=" * 60)

# --- Save StubLLM optimization traces ---
if opt_result.all_runs and opt_result.all_runs[0]:
    # Sample trace
    sample_otlp = opt_result.all_runs[0][0].otlp
    p = os.path.join(RUN_FOLDER, "stub_sample_otlp.json")
    with open(p, "w") as f:
        json.dump(sample_otlp, f, indent=2)
    print(f"  {p}")

    # All optimization traces
    all_traces = []
    for iter_idx, runs in enumerate(opt_result.all_runs):
        label = "baseline" if iter_idx == 0 else f"iteration_{iter_idx}"
        for ri, run in enumerate(runs):
            all_traces.append({
                "iteration": label,
                "query_index": ri,
                "score": run.score,
                "otlp": run.otlp,
            })
    p = os.path.join(RUN_FOLDER, "stub_all_traces.json")
    with open(p, "w") as f:
        json.dump(all_traces, f, indent=2)
    print(f"  {p}  ({len(all_traces)} traces)")

    # TGJ from first run
    tgj_docs = otlp_traces_to_trace_json(
        sample_otlp, agent_id_hint="m1-notebook", use_temporal_hierarchy=True,
    )
    p = os.path.join(RUN_FOLDER, "stub_sample_tgj.json")
    with open(p, "w") as f:
        json.dump(tgj_docs, f, indent=2)
    print(f"  {p}")

# --- Summary ---
summary = {
    "mode": "stub",
    "baseline_score": opt_result.baseline_score,
    "best_score": opt_result.best_score,
    "best_iteration": opt_result.best_iteration,
    "score_history": opt_result.score_history,
    "final_parameters": opt_result.final_parameters,
}
p = os.path.join(RUN_FOLDER, "stub_summary.json")
with open(p, "w") as f:
    json.dump(summary, f, indent=2)
print(f"  {p}")

# --- Save live traces if available ---
if HAS_API_KEY and 'live_opt_result' in dir():
    live_traces = []
    for iter_idx, runs in enumerate(live_opt_result.all_runs):
        label = "baseline" if iter_idx == 0 else f"iteration_{iter_idx}"
        for ri, run in enumerate(runs):
            live_traces.append({
                "iteration": label,
                "query_index": ri,
                "score": run.score,
                "otlp": run.otlp,
            })
    p = os.path.join(RUN_FOLDER, "live_all_traces.json")
    with open(p, "w") as f:
        json.dump(live_traces, f, indent=2)
    print(f"  {p}  ({len(live_traces)} traces)")

    live_summary = {
        "mode": "live",
        "model": OPENROUTER_MODEL,
        "baseline_score": live_opt_result.baseline_score,
        "best_score": live_opt_result.best_score,
        "score_history": live_opt_result.score_history,
        "final_parameters": live_opt_result.final_parameters,
        "total_llm_calls": live_llm.call_count,
    }
    p = os.path.join(RUN_FOLDER, "live_summary.json")
    with open(p, "w") as f:
        json.dump(live_summary, f, indent=2)
    print(f"  {p}")

print(f"\nAll artifacts saved to: {RUN_FOLDER}")

SAVING ARTIFACTS
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\stub_sample_otlp.json
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\stub_all_traces.json  (9 traces)
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\stub_sample_tgj.json
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\stub_summary.json
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\live_all_traces.json  (4 traces)
  H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1\live_summary.json

All artifacts saved to: H:\Freelance_Projects\Upwork\OTEL_Trace_Langraph\NewTrace_fork\examples\notebooks\notebook_outputs\m1


---
## Summary

This notebook demonstrated the full M1 pipeline:

1. **`instrument_graph()`** — one-liner to add OTEL tracing to a LangGraph
2. **`param.*` attributes** — spans carry trainable prompt values
3. **OTLP → TGJ → `ParameterNode` + `MessageNode`** — optimizer-compatible trace graph
4. **Temporal integrity** — child `gen_ai.*` spans don't break chaining
5. **`apply_updates()`** — bindings propagate optimizer output to live templates
6. **`optimize_graph()`** — end-to-end loop (StubLLM deterministic + live provider)
7. **Artifacts persisted** — OTLP JSON, TGJ JSON, and summaries saved to disk

All verifications passed with StubLLM (CI-safe, deterministic). When
`OPENROUTER_API_KEY` is set, the live section additionally proves
real-provider tracing with `param.*` and `gen_ai.*` attributes.